## Original section: Imports and configuration

**Teaching notes:** This cell imports the libraries used throughout the original classification and monitoring workflow. `scikit-learn` supplies the dataset, train/test split, classifier, and evaluation metrics; SHAP explains individual feature contributions; UMAP creates a lower-dimensional representation for embedding visualization; Faker creates synthetic metadata; and the Arize imports support prediction logging.

`RANDOM_STATE` makes the model and dimensionality-reduction steps reproducible. The notebook suppresses warnings for a cleaner classroom display, but in production you should inspect warnings rather than suppressing them globally.

In [ ]:
# Importing necessary libraries
import numpy as np
import uuid
import random
import umap
import shap
import pandas as pd
import faker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, accuracy_score
import datetime
from arize import ArizeClient

from arize.ml.types import (
    ModelTypes,
    Environments,
    Schema,
    Metrics,
    Embedding,
    EmbeddingColumnNames
)
import numpy as np
import pandas as pd
from faker import Faker


RANDOM_STATE = 123
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
from importlib.metadata import version

print("Arize version:", version("arize"))

## Original section: Helper functions

**Teaching notes:** These helpers create unique prediction IDs, synthetic event timestamps, and synthetic ages. The prediction ID is especially important: it is the stable key that lets a monitoring platform associate a prediction with its eventual actual outcome.

The timestamp helper creates event times across a requested interval. In a real application, use the time the prediction was actually made, not a randomly generated timestamp.

In [ ]:
def get_uuid():
    return uuid.uuid4().hex 
    
def random_dates(start, end, n=10):

    start_u = start.value//10**9
    end_u = end.value//10**9
    
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')


def random_age(min_age=0, max_age=100):
    """
    Generate a random age within the given range.
    Args:
        min_age (int): Minimum possible age.
        max_age (int): Maximum possible age.
    Returns:
        int: Randomly generated age.
    """
    if not (isinstance(min_age, int) and isinstance(max_age, int)):
        raise ValueError("min_age and max_age must be integers.")
    if min_age < 0 or max_age < 0:
        raise ValueError("Ages cannot be negative.")
    if min_age > max_age:
        raise ValueError("min_age cannot be greater than max_age.")

    return random.randint(min_age, max_age)

In [ ]:
fake = Faker()

## Original section: Load the breast-cancer dataset

**Teaching notes:** This is a supervised binary-classification problem. Each row contains numeric measurements derived from a breast-cancer sample, and `target` is the known outcome used for training and evaluation.

The target mapping is `0 → malignant` and `1 → benign` in scikit-learn's source dataset. Keep this mapping consistent when interpreting predicted labels, positive-class probabilities, confusion matrices, and monitoring metrics.

In [ ]:
# Load the breast cancer dataset
data = load_breast_cancer()

# Converting the dataset into a pandas DataFrame for easier handling
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # Add the target column (0 for malignant, 1 for benign)
df['target'] = df['target'].map({0: 'malignant', 1: 'benign'}) 

## Original section: Create prediction IDs and event metadata

**Teaching notes:** This cell adds the fields that make the dataset resemble production prediction events:

- `prediction_id`: unique identifier for each prediction record.
- `event_timestamp`: when the event occurred.
- `state`: synthetic categorical metadata that can be used for slicing or segmentation.

These metadata fields are not model inputs. They are useful for monitoring dimensions and record identity.

In [ ]:
# Create a primary key for the dataset
df['prediction_id'] = [get_uuid() for _ in range(df.shape[0])]

start = pd.to_datetime('2026-01-01')
end = pd.to_datetime('2026-02-28')

df['event_timestamp'] = random_dates(start, end, df.shape[0])

df['state'] = [fake.state() for _ in range(df.shape[0])]

## Original section: Add synthetic demographic / geographic tags

**Teaching notes:** `age` and `zip_code` are synthetic attributes added to demonstrate how metadata can be attached to model predictions and used to segment monitoring results.

They are deliberately excluded from the model's feature list below. In real systems, only collect and log attributes that are appropriate for the use case, permitted by policy, and safe to retain.

In [ ]:
df['age'] = [random_age() for _ in range(df.shape[0])]
df['zip_code'] = [fake.zipcode() for _ in range(df.shape[0])]

## Original section: Define features and create train/test sets

**Teaching notes:** The notebook separates model inputs from monitoring metadata. `features` contains only the breast-cancer measurements; prediction IDs, timestamps, state, age, and ZIP code are excluded from training.

The 80/20 split reserves a held-out test set for evaluation. The same feature columns and ordering must be used consistently during training, inference, and monitoring. For production evaluation, also consider stratifying the split so both classes are represented proportionally.

In [ ]:
# Splitting the dataset into training and testing sets (80% training, 20% testing)
X = df.drop('target', axis=1)  # Features
y = df['target']               # Target

features = X.drop(['prediction_id', 'event_timestamp', 'state', 'age', 'zip_code'], axis=1).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Original section: Train the Random Forest classifier

**Teaching notes:** A Random Forest combines many decision trees to produce a classification. The model is fit only on `X_train[features]` and `y_train`; the held-out test set is not used to fit the model.

This trained estimator becomes the source of the predictions, class probabilities, and SHAP explanations logged in the following sections.

In [ ]:
# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Fit the model to the training data
rf_classifier.fit(X_train[features], y_train)

## Original section: Evaluate classification predictions

**Teaching notes:** This is the baseline evaluation before sending records to Arize. Accuracy gives the overall fraction of correct predictions, while the classification report provides precision, recall, and F1-score for each class.

For a medical classification example, accuracy alone is not sufficient: false negatives and false positives have different implications. Students should inspect per-class recall and precision and connect those metrics to the business or clinical cost of each error.

In [ ]:
# Make predictions on the test set
y_pred = rf_classifier.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Original section: Explain predictions with SHAP

**Teaching notes:** SHAP estimates how each feature contributes to a model output relative to a baseline. Here, `TreeExplainer` is used because the Random Forest is tree-based.

The notebook selects the SHAP values for class index `1` and renames the feature columns with a `_shap` suffix. Those values can be uploaded alongside predictions so an analyst can investigate which features influenced model behavior.

**Discussion prompt:** How is a global feature-importance ranking different from the SHAP explanation for one individual prediction?

In [ ]:
# Create the SHAP explainer for the trained model
explainer = shap.TreeExplainer(rf_classifier)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test[features])

shap_values_positive_class = shap_values[:,:,1]

# 6. Convert the SHAP values to a DataFrame
shap_df = pd.DataFrame(shap_values_positive_class, columns=features)

# Because we are doing a bulk upload we need to rename the columns by added _shap at the end
shap_dataframe = pd.DataFrame(
        shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X_test.index
)

# Create a mapping
mapping = {fn: f"{fn}_shap" for fn in X[features].columns}

shap_dataframe.head()

## Original section: Create UMAP embeddings

**Teaching notes:** UMAP reduces the original feature space to three dimensions. The resulting coordinates provide a compact representation that can be visualized to explore clusters, overlap, and unusual groups of observations.

The embedding is fitted on the training data and later used to transform test and production-like records. This is important: fit the embedding once, then use the same fitted transformation for subsequent records so coordinates remain comparable.

In [ ]:
embedder = umap.UMAP(random_state=RANDOM_STATE, n_components=3)

X_train_embeddings = embedder.fit_transform(X_train[features], y_train.map({'benign': 0, 'malignant': 1}))

## Original section: Inspect the SHAP column mapping

**Teaching notes:** This mapping connects each original feature name to the corresponding SHAP-value column name. Arize uses this relationship to understand which explanation value belongs to which model feature.

When feature names change, update the mapping at the same time; mismatched names can make explanations confusing or unusable.

In [ ]:
mapping

## Original section: Define metadata tag columns

**Teaching notes:** These columns are intended for segmentation and filtering in monitoring views. Tags are descriptive attributes about the prediction context, rather than inputs used by the classifier.

For example, students can compare prediction behavior across states or age groups. The values in this notebook are synthetic and should not be interpreted as real demographic information.

In [ ]:
tag_columns = ['state', 'age', 'zip_code']

## Original section: Define the Arize data schema

**Teaching notes:** The schema tells Arize how to interpret the uploaded DataFrame:

- prediction ID and timestamp identify each event.
- tag columns define segmentation attributes.
- SHAP mapping links feature names to explanation values.
- prediction and actual label columns describe model outcomes.
- prediction score carries the model's score/probability.
- embedding configuration identifies the vector column.
- feature columns identify the model inputs.

The schema is the contract between the notebook's data and the monitoring platform. If the DataFrame column names change, the schema must be updated too.

In [ ]:
embedding_feature_column_names = {
    # Dictionary keys will be the name of the embedding feature in the app
    "embedding": EmbeddingColumnNames(
        vector_column_name='embedding', # column containing embedding vector (required)
        # data_column_name="text", # column containing raw text (optional NLP)
        # link_to_data_column_name="image_link" # column containing image URL links (optional CV)
    )
}


# define schema 
schema = Schema(
    prediction_id_column_name="prediction_id",
    timestamp_column_name="event_timestamp",
    tag_column_names=tag_columns,
    shap_values_column_names=mapping,
    prediction_label_column_name="predicted_label",
    actual_label_column_name="actual_label",
    prediction_score_column_name="prediction_score",
    embedding_feature_column_names=embedding_feature_column_names,
    feature_column_names=features.values.tolist(),
)


## Original section: Build a complete upload dataset

**Teaching notes:** This function assembles the monitoring payload for a batch of predictions. It starts with the original feature and metadata columns, then adds:

1. SHAP values for each observation.
2. The known actual label and model-predicted label.
3. The model's prediction score.
4. UMAP coordinates and the three-dimensional embedding vector.

This creates one row per prediction with both model outputs and the context needed for monitoring and analysis. Notice that the function receives the fitted model, explainer, and embedder so it can apply the same artifacts consistently.

In [ ]:
def create_upload_dataset(X, y_actual, y_pred, y_proba, features, model, explainer, embedder):
    df_test = X.copy()

    # Calculate SHAP values for the test set
    shap_values = explainer.shap_values(X[features])
    
    shap_values_positive_class = shap_values[:,:,1]
    
    # 6. Convert the SHAP values to a DataFrame
    shap_df = pd.DataFrame(shap_values_positive_class, columns=features)
    
    # Because we are doing a bulk upload we need to rename the columns by added _shap at the end
    shap_dataframe = pd.DataFrame(
            shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X.index
    )

    df_test = pd.concat([df_test, shap_dataframe], axis=1)
    df_test['actual_label'] = y_actual
    df_test['predicted_label'] = y_pred
    df_test['prediction_score'] = model.predict_proba(df_test[features])[:,1]
    df_test[['x','y','z']] = embedder.transform(df_test[features])
    df_test['embedding'] = df_test[['x','y','z']].values.tolist()
    
    df_test.reset_index(drop=True, inplace=True)

    return df_test


## Original section: Generate the validation batch

**Teaching notes:** This call uses the held-out test set to generate the complete upload DataFrame. Since the test labels are known, the records can include both predictions and actual outcomes.

This is validation data, not live production traffic. Keeping validation and production environments separate helps avoid mixing offline evaluation with operational monitoring.

In [ ]:
df_test = create_upload_dataset(X_test, y_test, rf_classifier.predict(X_test[features]), rf_classifier.predict_proba(X_test[features])[:,1],features, rf_classifier, explainer, embedder)

## Original section: Load credentials from the environment

**Teaching notes:** Credentials are read from environment variables or a local `.env` file. This avoids embedding secrets directly in notebook cells.

Before running the Arize examples, configure `ARIZE_API_KEY` and the appropriate space identifier. Never commit `.env` files or API keys to a class repository.

In [ ]:
import os
from dotenv import load_dotenv

# Put ARIZE_API_KEY and ARIZE_SPACE_ID in a local .env file or environment variables.
load_dotenv()

## Original section: Review the schema

**Teaching notes:** Displaying the schema is a quick checkpoint before ingestion. Ask students to verify that every configured column exists in the upload DataFrame and that the prediction, actual, score, timestamp, feature, tag, SHAP, and embedding fields have the intended meaning.

In [ ]:
schema

## Original section: Preview the upload records

**Teaching notes:** Inspecting the first few rows helps catch problems before sending data to Arize. Students should look for missing values, unexpected data types, incorrect class labels, and whether the SHAP and embedding columns were created.

A successful API request does not guarantee that the data is semantically correct, so this manual inspection is an important step.

In [ ]:
df_test.head()

## Original section: Upload a validation batch to Arize

**Teaching notes:** This cell creates the Arize client and sends the batch as a validation dataset. The batch ID makes the upload identifiable, while model ID and version identify the model that produced the predictions.

The classification metric configuration asks Arize to evaluate classification performance using the available prediction and actual labels. After ingestion, students can explore model performance and data behavior in the Arize UI.

**Instructor note:** The original code uses the legacy Pandas logger interface. The added extension later in this notebook demonstrates the newer SDK path; avoid mixing API styles in a single production implementation.

In [ ]:
df_test

In [ ]:
# create Arize client
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
API_KEY = os.getenv("ARIZE_API_KEY")

# Initialize the Arize client
arize_client = ArizeClient(api_key=API_KEY)

batch_id = f"VALIDATION-{get_uuid()}"
print(f"Uploading {batch_id}")

# Upload validation data as a batch
response = arize_client.ml.log(
    space_id=SPACE_ID,
    dataframe=df_test,
    schema=schema,
    model_name="binary-classification-5",
    model_version="1.0.0",
    model_type=ModelTypes.SCORE_CATEGORICAL,
    metrics_validation=[Metrics.CLASSIFICATION],
    validate=True,
    batch_id=batch_id,
    environment=Environments.VALIDATION
)

print("Upload successful!")
print("HTTP status:", response.status_code)

## Original section: Train a second model (K-Nearest Neighbors)

**Teaching notes:** KNN provides a contrasting classification algorithm. Instead of learning an ensemble of decision trees, it predicts a class based on the labels of nearby training observations.

Training this second model creates an opportunity to compare model behavior on the same dataset. In practice, scale numeric features before KNN because distance-based methods are sensitive to feature scale.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train[features], y_train)
# prediction = knn.predict(pd.DataFrame([feature_values]))

## Original section: Evaluate the KNN classifier

**Teaching notes:** The same held-out test set and classification metrics are used to evaluate KNN. Comparing this report with the Random Forest report illustrates that different algorithms can produce different precision/recall trade-offs even when trained on the same data.

For a fair comparison, keep the split and evaluation population fixed.

In [ ]:
# Make predictions on the test set
y_pred = knn.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Original section: Initialize the Arize client for online logging

**Teaching notes:** This client is used in the following single-record and streaming examples. Those examples demonstrate prediction-time logging rather than uploading a complete validation batch.

The key conceptual difference is timing: a production prediction is often logged before its actual outcome is known.

In [ ]:
# from arize.api import Client
#
# arize_client = Client(space_id=SPACE_ID, api_key=API_KEY)

## Original section: Create one synthetic production prediction

**Teaching notes:** This cell creates one prediction event with a unique ID, timestamp, and synthetic input features sampled within the observed feature ranges.

Because the feature values are randomly generated, this is a demonstration record rather than a real observation from the original dataset. It is useful for showing the mechanics of online inference and logging, but should not be used to draw conclusions about real-world model quality.

In [ ]:
import random

record = {
    'prediction_id': get_uuid()
    , 'state': fake.state()
    , 'event_timestamp': random_dates(start, end, 1).to_pydatetime()[0]
}

for column in X[features]:
    record.update({column: float(random.uniform(min(X[column].values), max(X[column].values)))})
    # break

## Original section: Score and log one production prediction

**Teaching notes:** This cell performs the production-style workflow for one record:

1. Select the model input features.
2. Generate SHAP explanations and a UMAP embedding.
3. Run classifier inference and obtain the predicted class and score.
4. Log the prediction, timestamp, model version, features, embedding, SHAP values, and tags to Arize.

Notice that this is a prediction event. Since the synthetic record has no known ground truth, it should not be assigned an actual label. When the real outcome becomes available, log it later using the same prediction ID.

In [ ]:
feature_values = {key: value for key, value in record.items() if key in features}

id = record['prediction_id']
print(f'logging {id}')

shap_values = explainer.shap_values(pd.DataFrame([feature_values]))
embedding = embedder.transform(pd.DataFrame([feature_values]))
embedding_payload = {
    "embedding": Embedding(
        vector = embedding.tolist()[0]
        # data: Union[str, List[str], NoneType] = None,
        # link_to_data: Optional[str] = None,
    )
}
label = rf_classifier.predict(pd.DataFrame([feature_values]))[0]
proba = rf_classifier.predict_proba(pd.DataFrame([feature_values]))[:,1][0]

response = arize_client.ml.log_stream(
    space_id=SPACE_ID,
    prediction_id=id,
    model_name="binary-classification-5",
    model_type=ModelTypes.SCORE_CATEGORICAL,
    environment=Environments.PRODUCTION,
    shap_values=dict(zip(feature_values.keys(), shap_values[:, :, 1][0])),
    model_version="1.0.0",
    prediction_timestamp=int(record["event_timestamp"].timestamp()),
    features=feature_values,
    embedding_features=embedding_payload,
    prediction_label=(label, proba),
    tags={
        "age": random_age(),
        "state": fake.state(),
        "zip_code": fake.zipcode()
    }
)

res = response.result()
if res.status_code == 200:
    print('Success sending Prediction!')
else:
    print(f'Log failed with response code {res.status_code}, {res.text}')

## Original section: Optional class inspection

**Teaching notes:** This commented expression lets students inspect the distinct target labels present in the dataset. It is a quick check of the classification problem's class vocabulary.

In [ ]:
# list(set(df['target'].values.tolist()))

## Original section: Optional class-balance check

**Teaching notes:** This commented expression calculates the fraction of rows belonging to one class. Class balance matters because accuracy can be misleading when one class is much more common than another.

Ask students to compare class prevalence with precision, recall, and the model's predicted class distribution.

In [ ]:
# sum(df['target'] == population[0]) / df.shape[0]

## Original section: Ground-truth logging reminder

**Teaching notes:** The synthetic record has no observed outcome, so the notebook intentionally does not log an actual label for it. This is a key monitoring concept: a prediction is not its own ground truth.

In production, retain the prediction ID and attach the actual outcome only after it is observed. This delayed-label pattern is also relevant to the regression example added later.

In [ ]:
# Do not log an actual label for this synthetic record.
# The record above contains randomly generated features and has no known ground truth.
# In production, log actual_label later only when the real outcome is observed,
# using the same prediction_id as the original prediction.

## Original section: Define the synthetic production time window

**Teaching notes:** The following loop simulates prediction events arriving over a short date range. In a real system, timestamps should come from the prediction service or event source, with consistent timezone handling.

In [ ]:
start = pd.to_datetime('2026-03-01')
end = pd.to_datetime('2026-03-04')

## Original section: Simulate a stream of production predictions

**Teaching notes:** This loop creates and logs 500 synthetic prediction events. Each iteration generates a new prediction ID, timestamp, and feature vector; computes the prediction, score, SHAP values, and embedding; then sends the record to Arize as a production event.

This demonstrates how model monitoring receives a stream of individual prediction records over time. It also creates data volume for exploring production dashboards and distributions.

**Discussion prompt:** What could go wrong if the feature-generation distribution in this simulation differs from the distribution seen by the model during training? How might that appear in drift monitoring?

Because the features are generated independently and uniformly within their observed ranges, the simulated stream is not a realistic representation of the joint feature distribution. Treat it as an ingestion/monitoring mechanics demo, not a realistic drift experiment.

In [ ]:
for _ in range(500):
    record = {
        'prediction_id': get_uuid()
        , 'state': fake.state()
        , 'event_timestamp': random_dates(start, end, 1).to_pydatetime()[0]
    }
    
    for column in X[features]:
        record.update({column: float(random.uniform(min(X[column].values), max(X[column].values)))})

    feature_values = {key: value for key, value in record.items() if key in features}

    id = record['prediction_id']
    print(f'logging {id}')
    
    shap_values = explainer.shap_values(pd.DataFrame([feature_values]))
    embedding = embedder.transform(pd.DataFrame([feature_values]))
    embedding_payload = {
        "embedding": Embedding(
            vector = embedding.tolist()[0]
            # data: Union[str, List[str], NoneType] = None,
            # link_to_data: Optional[str] = None,
        )
    }
    label = rf_classifier.predict(pd.DataFrame([feature_values]))[0]
    proba = rf_classifier.predict_proba(pd.DataFrame([feature_values]))[:,1][0]
    
    response = arize_client.ml.log_stream(
            prediction_id=id,
            space_id=SPACE_ID,
            model_name="binary-classification-5",
            model_type=ModelTypes.SCORE_CATEGORICAL,
            environment=Environments.PRODUCTION,
            shap_values=dict(zip(feature_values.keys(), shap_values[:,:,1][0])),
            model_version='1.0.0',
            prediction_timestamp=int(record['event_timestamp'].timestamp()),
            features=feature_values,
            embedding_features=embedding_payload,
            prediction_label=(label, proba),
            tags={
                'age': random_age()
                , 'state': fake.state()
                , 'zip_code': fake.zipcode()
            }
    )
    
    res = response.result()
    if res.status_code == 200:
        print(f'Success sending Prediction for {id}')
    else:
        print(f'Log failed with response code {res.status_code}, {res.text}')



## How the original sections connect to the new extension

The first part of this notebook teaches batch validation logging and production-style prediction logging for classification. The extension builds on those ideas by adding a regression model and an instrumented agent workflow.

The common monitoring concepts are **model identity, model version, timestamp, prediction ID, features, prediction, and (when available) actual outcome**. The agent trace adds execution context—agent runs, LLM calls, and tool spans—while the ML monitoring records continue to hold structured model inputs, predictions, and outcomes.

# Extension: ML Model Monitoring + Agent Tracing in Arize AX

This extension adds two connected observability paths:

1. **ML model monitoring:** log ordinary classifier and regression predictions, features, prediction IDs, and ground-truth actuals.
2. **Agent observability:** instrument an OpenAI Agents SDK workflow and export its agent/tool/model spans to Arize AX using OpenTelemetry and OpenInference.

The agent calls the ML models as tools. The same `prediction_id` is attached to the tool span and used when logging the corresponding ML prediction. This gives us a practical correlation key between the agent trace and the ML model record.

> Run this section after the original notebook's classifier training cells. It uses `rf_classifier`, `X_test`, `y_test`, and `features` from the earlier cells. It creates a separate regression model using scikit-learn's diabetes dataset.

## 1. Install the current SDKs

The original notebook uses the older `arize.pandas.logger` / `arize.api` interfaces. This extension reuses the prediction logger already initialized in the original notebook and adds `arize-otel` plus OpenInference for tracing.

If you are in a managed class environment, install these packages in the environment before running the notebook. A kernel restart may be required.

In [ ]:
%pip install -U arize-otel openai-agents openinference-instrumentation-openai-agents scikit-learn python-dotenv

## 2. Configure Arize AX and the model-monitoring client

Set `ARIZE_SPACE_ID` and `ARIZE_API_KEY` in your environment or `.env` file. Do not commit credentials to the notebook or source control.

The `project_name` used by tracing is the Arize AX project that will contain the agent traces. ML model IDs are separate identifiers used by the ML monitoring experience.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Reuse the legacy Arize client initialized in the original notebook (cell 23).
# The original notebook's SPACE_ID and API_KEY values come from its setup cells.
# Prefer environment variables over hard-coded credentials.
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
API_KEY = os.getenv("ARIZE_API_KEY")

REGRESSION_MODEL_NAME = "diabetes-regression"
CLASSIFIER_MODEL_NAME = "breast-cancer-random-forest"
MODEL_VERSION = "1.0.0"

print("Reusing the original Arize prediction logger.")
from datetime import datetime, timezone

## 3. Train a regression model

The breast-cancer example above demonstrates classification. This section adds a regression use case using scikit-learn's diabetes dataset.

For regression, the model output and the actual target are continuous numeric values. We will monitor regression-specific performance (for example, MAE, RMSE, and R²) rather than classification accuracy.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import uuid
from datetime import datetime, timezone

regression_data = load_diabetes(as_frame=True)
X_reg = regression_data.data
y_reg = regression_data.target

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=123
)

regressor = RandomForestRegressor(
    n_estimators=150, min_samples_leaf=2, random_state=123
)
regressor.fit(X_reg_train, y_reg_train)

reg_predictions = regressor.predict(X_reg_test)

print(f"MAE:  {mean_absolute_error(y_reg_test, reg_predictions):.3f}")
print(f"RMSE: {mean_squared_error(y_reg_test, reg_predictions) ** 0.5:.3f}")
print(f"R²:   {r2_score(y_reg_test, reg_predictions):.3f}")

## 4. Log ordinary ML predictions and actuals to Arize

Each record needs a stable `prediction_id` so a later-arriving actual can be joined to its prediction. For this classroom demonstration, we log prediction and actual together because the held-out labels are already available.

In a real production system, log the prediction when it is made, then log the actual when the outcome becomes known. Do not send a model's own prediction as the actual label.

In [ ]:
def log_regression_record(model_input, prediction, actual, prediction_id=None, timestamp=None):
    """Log one regression prediction and its observed ground truth."""
    prediction_id = prediction_id or uuid.uuid4().hex
    timestamp = timestamp or int(datetime.now(timezone.utc).timestamp())

    response = arize_client.ml.log_stream(
        prediction_id=prediction_id,
        space_id=SPACE_ID,
        model_name=REGRESSION_MODEL_NAME,
        model_type=ModelTypes.NUMERIC,
        environment=Environments.PRODUCTION,
        model_version=MODEL_VERSION,
        prediction_timestamp=timestamp,
        prediction_label=float(prediction),
        actual_label=float(actual),
        features={k: float(v) for k, v in model_input.items()},
    )
    result = response.result()
    if result.status_code != 200:
        raise RuntimeError(f"Arize regression logging failed: {result.status_code} {result.text}")
    return prediction_id, result

### 4a. Log classifier predictions and actuals

The original notebook already demonstrates batch and single-record classifier logging with the legacy logger. The following uses the same single-record logger pattern as the original notebook.

For a binary model, `prediction_label` is supplied as `(class_label, positive_class_score)`. Confirm the class/score convention against the model configuration in your Arize space before using this pattern with a different binary model.

In [ ]:
def log_classifier_record(model_input, predicted_label, positive_score, actual_label,
                          prediction_id=None, timestamp=None):
    prediction_id = prediction_id or uuid.uuid4().hex
    timestamp = timestamp or int(datetime.now(timezone.utc).timestamp())

    response = arize_client.ml.log_stream(
        prediction_id=prediction_id,
        space_id=SPACE_ID,
        model_name=CLASSIFIER_MODEL_NAME,
        model_type=ModelTypes.SCORE_CATEGORICAL,
        environment=Environments.PRODUCTION,
        model_version=MODEL_VERSION,
        prediction_timestamp=timestamp,
        prediction_label=(str(predicted_label), float(positive_score)),
        actual_label=str(actual_label),
        features={k: float(v) for k, v in model_input.items()},
    )
    result = response.result()
    if result.status_code != 200:
        raise RuntimeError(f"Arize classifier logging failed: {result.status_code} {result.text}")
    return prediction_id, result

## 5. Configure agent traces and spans

Arize AX tracing uses OpenTelemetry transport and OpenInference semantic conventions. An agent execution becomes a trace tree; agent, LLM, and tool operations appear as spans. The OpenAI Agents SDK instrumentor automatically captures supported agent-run operations.

The tracing project is named `ml-monitoring-agent-demo`. This is distinct from the two ML model names above.

In [ ]:
from arize.otel import register
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

TRACE_PROJECT_NAME = "ml-monitoring-agent-demo"

tracer_provider = register(
    space_id=SPACE_ID,
    api_key=API_KEY,
    project_name=TRACE_PROJECT_NAME,
)

OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

print(f"Tracing configured for Arize project: {TRACE_PROJECT_NAME}")

## 6. Create an agent that invokes the ML models

The agent has one tool, `run_ml_models`, that runs one breast-cancer classification and one diabetes regression prediction. This keeps the example deterministic and avoids requiring a separate production API.

The tool creates one `prediction_id` per model prediction, logs that ID with the ML record, and attaches it to the active tool span as `ml.prediction_id`. In Arize, inspect the agent trace and then use that ID to locate the corresponding ML model record.

The OpenAI Agents SDK requires `OPENAI_API_KEY`. Agent tracing is separate from the Arize API key.

In [ ]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

# Select a provider: "groq", "gemini", or "openrouter"
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq")

PROVIDERS = {
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "gemini": {
        "api_key": os.getenv("GEMINI_API_KEY"),
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "model": "gemini-3.8-flash",
    },
    "openrouter": {
        "api_key": os.getenv("OPENROUTER_API_KEY"),
        "base_url": "https://openrouter.ai/api/v1",
        # Replace with an available model ending in :free
        "model": os.getenv(
            "OPENROUTER_MODEL",
            "google/gemma-4-26b-a4b-it:free"
        ),
    },
}

config = PROVIDERS[LLM_PROVIDER]

if not config["api_key"]:
    raise EnvironmentError(
        f"Set the API key for {LLM_PROVIDER} before running the agent."
    )

llm_client = AsyncOpenAI(
    api_key=config["api_key"],
    base_url=config["base_url"],
)

agent_model = OpenAIChatCompletionsModel(
    model=config["model"],
    openai_client=llm_client,
)

print(f"Using {LLM_PROVIDER}: {config['model']}")

In [ ]:
import json
from agents import Agent, Runner, function_tool
from opentelemetry import trace as trace_api

@function_tool
def run_ml_models(sample_index: int) -> str:
    """Run both demo ML models for a held-out sample index and log the predictions."""
    if sample_index < 0 or sample_index >= min(len(X_test), len(X_reg_test)):
        return json.dumps({"error": "sample_index is outside the demo sample range"})

    # Classification sample
    x_class = X_test.iloc[sample_index][features]
    class_frame = pd.DataFrame([x_class], columns=features)
    class_label = str(rf_classifier.predict(class_frame)[0])
    class_probabilities = rf_classifier.predict_proba(class_frame)[0]
    malignant_index = list(rf_classifier.classes_).index("malignant")
    class_score = float(class_probabilities[malignant_index])
    class_actual = str(y_test.iloc[sample_index])

    # Regression sample
    x_reg = X_reg_test.iloc[sample_index]
    reg_frame = pd.DataFrame([x_reg], columns=X_reg_test.columns)
    reg_prediction = float(regressor.predict(reg_frame)[0])
    reg_actual = float(y_reg_test.iloc[sample_index])

    # Link IDs from the ML records to this tool span.
    class_prediction_id = uuid.uuid4().hex
    reg_prediction_id = uuid.uuid4().hex

    active_span = trace_api.get_current_span()
    if active_span and active_span.is_recording():
        active_span.set_attribute("ml.classifier.prediction_id", class_prediction_id)
        active_span.set_attribute("ml.regression.prediction_id", reg_prediction_id)
        active_span.set_attribute("ml.sample_index", int(sample_index))

    log_classifier_record(
        x_class.to_dict(), class_label, class_score, class_actual,
        prediction_id=class_prediction_id
    )
    log_regression_record(
        x_reg.to_dict(), reg_prediction, reg_actual,
        prediction_id=reg_prediction_id
    )

    return json.dumps({
        "sample_index": int(sample_index),
        "classification": {
            "prediction": class_label,
            "malignant_probability": class_score,
            "actual": class_actual,
            "prediction_id": class_prediction_id,
        },
        "regression": {
            "prediction": reg_prediction,
            "actual": reg_actual,
            "prediction_id": reg_prediction_id,
        },
    }, indent=2)

ml_observability_agent = Agent(
    name="ML Monitoring Teaching Agent",
    instructions=(
        "You are a teaching assistant demonstrating model monitoring. "
        "When asked to run the models, call run_ml_models with a sample_index "
        "between 0 and 24. Explain the classifier output and the regression "
        "estimate. Clearly distinguish predictions from actual observed values."
    ),
    tools=[run_ml_models],
    model=agent_model,
)

## 7. Run the agent and generate traces

Each call to `Runner.run_sync` should produce an agent trace. Expand the trace in Arize AX to inspect the agent run, model/LLM activity, and tool invocation. Open the tool span to see the custom ML prediction IDs.

Try several prompts and compare traces:
- `Run both models for sample 3 and explain the outputs.`
- `Run both models for sample 12. Which prediction was farther from its actual value?`
- `Run both models for sample 20 and summarize the two prediction IDs.`

In [ ]:
agent_result = await Runner.run(
    ml_observability_agent,
    "Run both models for sample 3 and explain the predictions versus actual values."
)

print(agent_result.final_output)

# Flush batched spans so they are sent to Arize
tracer_provider.force_flush()

## 8. What to inspect in Arize AX

### ML model monitoring

Open the ML model monitoring area and inspect:
- `breast-cancer-random-forest`
- `diabetes-regression`
- Prediction volume and feature distributions
- Prediction versus actual performance
- Classification metrics for the binary classifier
- Regression metrics such as MAE, RMSE, and R²
- Performance by model version and time window

**Create monitors in the Arize UI after the first records arrive.** For each model, open its monitoring/metrics configuration and create monitors appropriate to the model type:
- Both models: prediction volume and feature drift / data quality.
- Classifier: prediction distribution and classification performance (for example, accuracy, precision, recall, or F1 once actuals are available).
- Regressor: regression performance (for example, MAE or RMSE) once actuals are available.

Choose thresholds from the model's baseline and business tolerance. The notebook sends telemetry; it does not automatically create dashboard monitors or alert policies in the Arize UI.

### Agent observability

Open the `ml-monitoring-agent-demo` tracing project:
- Find the trace generated by the prompt above.
- Expand the Agent span and its child spans.
- Inspect the tool span and its `ml.*.prediction_id` attributes.
- Compare the trace's latency and errors with the ML model record's prediction and actual.

### Important distinction

**ML monitoring and agent tracing are different telemetry products.** ML model records contain structured features, predictions, actuals, and model-performance signals. Agent traces contain the execution path, span hierarchy, inputs/outputs, latency, and tool/model interactions. A shared prediction ID is the correlation mechanism in this teaching example; it does not automatically merge the two datasets into one record.

## 9. Production considerations and student discussion

1. **Delayed actuals:** In production, log predictions first and actuals later using the same `prediction_id`. The earlier synthetic 500-record stream has no ground truth, so it now logs predictions only; the held-out classification and regression examples log actuals because their labels are known.
2. **Data privacy:** Do not capture raw PII, secrets, credentials, or sensitive customer data in span inputs/outputs. Apply redaction before exporting traces.
3. **Sampling:** High-volume agent applications may need trace sampling and bounded retention.
4. **Model identity:** Keep model name and version stable and meaningful. A model version should change when the deployed model artifact or relevant prediction behavior changes.
5. **Monitoring design:** Set alert thresholds based on business risk and baseline behavior, not arbitrary universal values.
6. **Correlation:** Propagate stable business/request IDs through agent, tool, and model-serving boundaries. Avoid putting sensitive identifiers into trace attributes.
7. **Validation vs. production:** Use validation datasets/batches for offline checks; use production environment for live prediction streams. Do not mix the two.